# Early stopping on the digits dataset

Trains the same MLP with and without early stopping and compares accuracy and training time.


In [ ]:
import tensorflow as tf
# Allocate GPU memory only as needed. By default TensorFlow grabs all of it, which makes a second
# notebook (or kernel) fail with an out-of-memory error on the 4 GB laptop GPU.
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)


In [ ]:
import time
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

X, y = load_digits(return_X_y=True)
X = X / 16.0
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# WITHOUT early stopping
model1 = keras.Sequential([Dense(64, activation='relu', input_shape=(64,)), Dense(10, activation='softmax')])
model1.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
t = time.time()
h1 = model1.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, verbose=0)
time1 = time.time() - t

# WITH early stopping
model2 = keras.Sequential([Dense(64, activation='relu', input_shape=(64,)), Dense(10, activation='softmax')])
model2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
t = time.time()
h2 = model2.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, callbacks=[stop], verbose=0)
time2 = time.time() - t

print('Time:', time1, 'vs', time2)
print('Epochs:', len(h1.history['loss']), 'vs', len(h2.history['loss']))
print('Val acc:', h1.history['val_accuracy'][-1], 'vs', h2.history['val_accuracy'][-1])
print('Overfit gap:', h1.history['accuracy'][-1] - h1.history['val_accuracy'][-1],
      'vs', h2.history['accuracy'][-1] - h2.history['val_accuracy'][-1])

plt.plot(h1.history['val_loss']); plt.plot(h2.history['val_loss'])
plt.legend(['Without ES', 'With ES']); plt.show()

In [ ]:
import time
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

X, y = load_digits(return_X_y=True)
X_train, X_val, y_train, y_val = train_test_split(X / 16, y, test_size=0.2)

# without early stopping
m1 = keras.Sequential([Dense(64, activation='relu'), Dense(10, activation='softmax')])
m1.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
t = time.time()
h1 = m1.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, verbose=0)
print('Without ES:', time.time() - t, 'sec', len(h1.history['loss']), 'epochs', h1.history['val_accuracy'][-1])

# with early stopping
m2 = keras.Sequential([Dense(64, activation='relu'), Dense(10, activation='softmax')])
m2.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
t = time.time()
h2 = m2.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, verbose=0,
            callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
print('With ES:', time.time() - t, 'sec', len(h2.history['loss']), 'epochs', h2.history['val_accuracy'][-1])

plt.plot(h1.history['val_loss']); plt.plot(h2.history['val_loss']); plt.legend(['Without', 'With']); plt.show()